In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

[回路への命令の可視化](/guides/visualize-circuits)に加えて、Qiskit の [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) メソッドを使用して回路のスケジューリングを可視化することもできます。この可視化は、例えば量子ビット上のアイドル時間をすばやく発見するのに役立ちます。ただし、このメソッドは動的回路に対して正確な結果を返しません。動的回路のスケジューリングを可視化するには、[Qiskit Runtime サポート](#qr-support) セクションで説明している `draw_circuit_schedule_timing` メソッドを使用してください。

## 例

スケジュールされた回路プログラムを可視化するには、一連の制御引数を指定してこの関数を呼び出します。出力画像の外観のほとんどはスタイルシートで変更できますが、必須ではありません。

### デフォルトのスタイルシートで描画する

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![前のコードセルの出力](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### プログラムデバッグに適したスタイルシートで描画する

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![前のコードセルの出力](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

カスタムのジェネレーター関数またはレイアウト関数を作成し、カスタム関数で既存のスタイルシートを更新することができます。これにより、スケジュールされた回路ドロワーのコードベースを変更することなく、出力画像の外観のほとんどを制御できます。詳細な例については、[`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) の API リファレンスを参照してください。

<span id="qr-support"></span>

## Qiskit Runtime サポート

Qiskit に組み込まれているタイムライン描画機能は静的回路には有用ですが、ブロードキャストやブランチ決定などの暗黙的な操作のために、[動的回路](/guides/classical-feedforward-and-control-flow)のタイミングを正確に反映しない場合があります。動的回路サポートの一環として、Qiskit Runtime はリクエストされた場合にジョブ結果の中に正確な回路タイミング情報を返します。

> **Note:** - これは実験的な機能です。プレビューリリース状態にあるため、変更される可能性があります。
> - この機能は Qiskit Runtime Sampler ジョブにのみ適用されます。
> - 回路の合計時間は「compilation」メタデータに返されますが、これは課金（量子時間）に使用される時間ではありません。

### タイミングデータの取得を有効にする

タイミングデータの取得を有効にするには、プリミティブジョブを実行する際に実験的な `scheduler_timing` フラグを `True` に設定します。

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### 回路タイミングデータへのアクセス
リクエストされた場合、各 PUB の回路タイミングデータはジョブ結果のメタデータ内の `["compilation"]["scheduler_timing"]["timing"]` に返されます。このフィールドには生のタイミング情報が含まれています。タイミング情報を表示するには、[タイミングの可視化](#visualize-timings) セクションで説明している組み込みの可視化ツールを使用してください。

最初の PUB の回路タイミングデータにアクセスするには、次のコードを使用します。

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### 生のタイミングデータを理解する
`draw_circuit_schedule_timing` メソッドを使用して回路タイミングデータを可視化することが最も一般的なユースケースですが、返される生のタイミングデータの構造を理解しておくと役立つ場合があります。これにより、例えばプログラムで情報を抽出することが可能になります。

`["compilation"]["scheduler_timing"]["timing"]` に返されるタイミングデータは文字列のリストです。各文字列は特定のチャネル上の単一の命令を表し、次のデータ型をカンマ区切りで構成しています。

- `Branch` - 命令がコントロールフロー（then / else）内にあるか、メインブランチにあるかを判定します。
- `Instruction` - ゲートと操作対象の量子ビット。
- `Channel` - 命令に割り当てられるチャネル。以下のいずれかになります。
      - `Qubit x` - 量子ビット _x_ のドライブチャネル。
      - `AWGRx_y`（任意波形ジェネレーター読み出し）- 量子ビット測定時の通信に読み出しチャネルが使用します。引数 _x_ と _y_ はそれぞれ読み出し機器 ID と量子ビット番号に対応します。
- `T0` - スケジュール全体における命令の開始時刻。
- `Duration` - 命令の持続時間（_dt_ 秒単位、1 dt = 1 スケジューリングサイクル）。バックエンドの `dt` 値は [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt) を使用して確認できます。
- `Pulse` - 使用されるパルス操作の種類。

例:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### タイミングの可視化
`qiskit-ibm-runtime` v0.43.0 以降では、回路タイミングを可視化できます。タイミングを可視化するには、まず [`draw_circuit_schedule_timing` メソッド](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26)を使用して結果メタデータを `fig` に変換する必要があります。このメソッドは `plotly` の図を返し、直接表示、ファイルへの保存、またはその両方が可能です。使用する `plotly` コマンドの詳細については、[`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) および [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html) を参照してください。

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![出力にカーソルを合わせると、開始時刻・終了時刻・持続時間などの情報が表示されます。](../docs/images/guides/visualize-circuit-timing/image_1.svg '生成された図の例')

#### 生成された図を理解する
`draw_circuit_schedule_timing` が出力する回路タイミングデータの画像には、次の情報が含まれています。

- X 軸は _dt_ 秒単位の時間です（1 dt = 1 スケジューリングサイクル）。バックエンドの `dt` 値は [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt) を使用して確認できます。
- Y 軸はチャネルです（チャネルはパルスを放出する機器と考えてください）。
    - `Receive channel` - それ自体は機器ではない唯一のチャネルです。そのとき Hub との通信手順に関わるすべてのチャネルで再生される命令です。
    - `Qubit x` - 量子ビット x のドライブチャネル。
    - `AWGRx_y`（任意波形ジェネレーター読み出し）- 量子ビット測定時の通信に読み出しチャネルが使用します。引数 _x_ と _y_ はそれぞれ読み出し機器 ID と量子ビット番号に対応します。
    - `Hub` - ブロードキャストを制御します。

また、各命令は *X_Y* の形式で表されます。ここで *X* は命令名、*Y* はパルスタイプです。`play` タイプは制御パルスを適用し、`capture` は量子ビットの状態を記録します。各命令にカーソルを合わせると詳細情報も確認できます。例えば、上記の図では量子ビット 10 に対して 1161 dt の時点で X ゲートの制御パルスが適用されていることが示されています。

### エンドツーエンドの例
この例では、オプションを有効にし、メタデータから回路タイミング情報を取得して、画像として表示する方法を示します。

まず、環境を設定し、回路を定義して ISA 回路に変換し、ジョブを定義・実行します。

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

次に、回路スケジュールタイミングを取得します。